# Phase 5 — Backtest results

Strategy vs buy-and-hold on the test split only (never val/train).  
Costs: 0.1% taker fee + 0.1% slippage per trade side.

In [ ]:
import matplotlib.pyplot as plt

from src.backtest.report import load_backtest_config, load_modeling_config, run_backtest

cfg = load_modeling_config("configs/config.yaml")
bt_cfg = load_backtest_config("configs/config.yaml")
results = run_backtest("1d", cfg, bt_cfg, model_variant="pruned")

lr  = results["lr"]
lgb = results["lgb"]
bah = results["buyhold"]
print(f"LR  total return: {lr.total_return_pct:.2f}%  ({lr.n_trades} trades)")
print(f"LGB total return: {lgb.total_return_pct:.2f}%  ({lgb.n_trades} trades)")
print(f"B&H total return: {bah.total_return_pct:.2f}%  ({bah.n_trades} trades)")

## Equity curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

notional = lr.starting_notional
for res, color, ls in [
    (lr,  "steelblue",  "-"),
    (lgb, "darkorange", "--"),
    (bah, "gray",       "-."),
]:
    (res.equity_curve / notional * 100).plot(
        ax=ax, label=f"{res.label} ({res.total_return_pct:+.1f}%)",
        color=color, linestyle=ls,
    )

ax.axhline(100, color="black", linewidth=0.6, linestyle=":")
ax.set_ylabel("Equity (index, start=100)")
ax.set_title("BTCUSDT 1d — equity curves (test split, 2025-08-17 → 2026-08-11)")
ax.legend()
fig.tight_layout()
plt.show()

## Drawdown

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for res, color, ls in [
    (lr,  "steelblue",  "-"),
    (lgb, "darkorange", "--"),
    (bah, "gray",       "-."),
]:
    eq = res.equity_curve
    dd = (eq / eq.cummax() - 1) * 100
    dd.plot(
        ax=ax,
        label=f"{res.label} (max DD {res.max_drawdown_pct:.1f}%)",
        color=color, linestyle=ls,
    )

ax.set_ylabel("Drawdown (%)")
ax.set_title("Drawdown from peak")
ax.legend()
fig.tight_layout()
plt.show()

## Comparison table

In [ ]:
from src.backtest.report import comparison_table

print(comparison_table([lr, lgb, bah]))

## Honest reading

BTC fell roughly **−46%** during the test window (Aug 2025 → Aug 2026).

* **LGB** appears to outperform buy-and-hold (−33% vs −46%) but only because
  it fired just **30 signals** in 328 days — 91% cash.  Being mostly absent
  from a crashing market is not the same as having directional edge.
  In a bull market the same low-coverage model would massively underperform.

* **LR** underperforms buy-and-hold (−49% vs −46%) despite 16 short signals:
  its 49 long signals landed in downtrend windows, and 53 round-trips consumed
  ~10% in fees.

* **Sharpe** is negative for all three, which is the expected result for any
  system operating in a sustained downtrend.

**Conclusion:** no model should be traded with real money based on these results.
Phase 4 found no directional edge; Phase 5 confirms real costs make the absence
of edge fatal once friction is applied.